
# GRDP and GRDP Per Capita Profiling
Owner: Razz
Tables:
- `pondolens.dev_bronze.grdp`
- `pondolens.dev_bronze.grdp_per_capita`

Baseline: 114 rows per table

## Goal
Verify economic measures and compatibility with Q3. Do not modify Bronze.

## Checks
- Compare Excel source counts with Bronze; exclude headers.
- Check column types, nulls/blanks, and sample values against the source.
- Verify uniqueness by region, year, price basis, and geographic level.
- Confirm 19 entries per year/price basis and identify national versus regional records.
- Verify region names/IDs, years, numeric amounts, and units in both tables.
- Keep current prices and constant 2018 prices separate.
- Check matching regions and boundaries with population and regional spending.
- Confirm joins do not multiply records.

## How to read the result
The cell below shows 2 tables.
- **Source Check**: one row per check and table. `Rows Checked` is how many rows were looked at, `Bronze` is the row count in Bronze, `Source` is the same row count in the original Volume. Counts are rows affected, so 0 is good (except `Table Row Count`). The last rows of each table compare Bronze with the source file: the `Source` cell is empty and `Bronze` is how many rows differ.
- **Data Types**: each Bronze column and its current data type.


In [0]:
import importlib
import subprocess
import sys

import pandas as pd
import numpy as np

# pandas needs openpyxl to read the excel files; install it if the cluster does not have it
try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])
    importlib.invalidate_caches()

BRONZE = {
    "grdp": "pondolens.dev_bronze.grdp",
    "grdp_per_capita": "pondolens.dev_bronze.grdp_per_capita",
}

# original excel files that were loaded into bronze
SOURCE_PATH = "/Volumes/pondolens/dev_bronze/government_budget/source/05_development_indicators/"
SOURCE = {
    "grdp": SOURCE_PATH + "grdp_2023_2025.xlsx",
    "grdp_per_capita": SOURCE_PATH + "grdp_per_capita_2023-2025.xlsx",
}

MEASURE = {"grdp": "GRDP_amount", "grdp_per_capita": "GRDP_per_capita_amount"}
KEY = ["Region", "year", "price_basis", "geographic_level"]
JOIN_KEY = ["Region", "price_basis", "year"]
DASHES = ["-", "--", "–", "—"]


def read_source(path):
    try:
        df = pd.read_excel(path, keep_default_na=False, na_values=[""])
    except Exception as e:
        print("could not read source file:", path, "-", e)
        return None
    return df


bronze, source, types = {}, {}, {}
for name, table in BRONZE.items():
    sdf = spark.table(table)
    bronze[name] = sdf.toPandas()
    types[name] = dict(sdf.dtypes)
    source[name] = read_source(SOURCE[name])


# each count is the number of rows affected, not cells
def profile(df):
    blank = lambda s: s.isna() | (s.astype(str).str.strip() == "")
    zero = df.apply(lambda c: pd.to_numeric(c, errors="coerce") == 0)
    dash = df.apply(lambda c: c.notna() & c.astype(str).str.strip().isin(DASHES))
    return {
        "Table Row Count": len(df),
        "Region ID Empty Count": blank(df["region_id"]).sum(),
        "Region Empty Count": blank(df["Region"]).sum(),
        "Null Count": df.isna().any(axis=1).sum(),
        "Zero Count": zero.any(axis=1).sum(),
        "Dash Count": dash.any(axis=1).sum(),
        "Exact Duplicate Rows": df.duplicated(keep=False).sum(),
        "Composite Key Duplicates (Region, Year, Price Basis, Geographic Level)": df.duplicated(KEY, keep=False).sum(),
    }


def same(a, b):
    return (a == b) | (a.isna() & b.isna())


rows = []
for name in BRONZE:
    brz, src = bronze[name], source[name]
    for col in ["year", "source_row", MEASURE[name]]:
        brz[col] = pd.to_numeric(brz[col], errors="coerce")
        if src is not None:
            src[col] = pd.to_numeric(src[col], errors="coerce")

    b = profile(brz)
    s = profile(src) if src is not None else {}
    for check, value in b.items():
        rows.append([name, check, len(brz), value, s.get(check)])

    # bronze vs source: the same row is matched on Region + price_basis + year
    if src is None:
        continue
    m = src.merge(brz, on=JOIN_KEY, how="outer", indicator=True, suffixes=("_src", "_brz"))
    both = m[m["_merge"] == "both"]
    rows.append([name, "Rows In Source Not In Bronze", len(src),
                 (m["_merge"] == "left_only").sum(), None])
    rows.append([name, "Rows In Bronze Not In Source", len(brz),
                 (m["_merge"] == "right_only").sum(), None])
    for col, label in [("region_id", "Region ID"), ("geographic_level", "Geographic Level"), ("unit", "Unit")]:
        diff = ~same(both[col + "_src"].astype("object"), both[col + "_brz"].astype("object"))
        rows.append([name, f"{label} Mismatch", len(both), diff.sum(), None])
    amount = MEASURE[name]
    diff = ~np.isclose(both[amount + "_src"].astype(float), both[amount + "_brz"].astype(float),
                       rtol=1e-9, equal_nan=True)
    rows.append([name, f"{amount} Mismatch", len(both), diff.sum(), None])

checks = pd.DataFrame(rows, columns=["Table", "Check", "Rows Checked", "Bronze", "Source"])
for col in ["Rows Checked", "Bronze", "Source"]:
    checks[col] = checks[col].astype("Int64")

column_types = pd.DataFrame(
    [[name, col, dtype] for name in BRONZE for col, dtype in types[name].items()],
    columns=["Table", "Column", "Bronze Data Type"])

print("Source Check")
display(checks)

print("Data Types")
display(column_types)

## Findings
Result of the checks on the Bronze tables, compared with the original Excel files in the Volume. Both tables have 114 rows.

| Table | Issue | Affected rows | Rows checked | Proposed Silver treatment |
|---|---|---|---|---|
| grdp | `region_id` is empty (MIMAROPA Region 6, Philippines 6) | 12 | 114 | Fill the code: MIMAROPA Region = `IV-B` and Philippines = `PH`, 'all', or null. Needs to be reviewed. |
| grdp_per_capita | `region_id` is empty (MIMAROPA Region 6, Philippines 6) | 12 | 114 | Fill the code: MIMAROPA Region = `IV-B` and Philippines = `PH`, 'all', or null. Needs to be reviewed. |

- Table Row Count is 114 in Bronze and in the source volume.
- Region Empty, Zero, Dash, Exact Duplicate Rows and Composite Key Duplicates (Region, Year, Price Basis, Geographic Level) are all 0.
- The 12 rows with a null are the same 12 rows with an empty `region_id`. The source file has the same 12.
- Bronze rows matches the source file rows.
- 0 differences in `region_id`, `geographic_level`, `unit` and the amount.

## Data Types (Bronze)
| Table | Observation | Proposed Silver treatment |
|---|---|---|
| grdp, grdp_per_capita | `source_row` and `year` are `bigint`. Text columns are `string`. | Cast `source_row` and `year` to INT. |
| grdp | `GRDP_amount` is `decimal(19,8)` | Keep as decimal or cast to DOUBLE. Current values fit with room to grow. |
| grdp_per_capita | `GRDP_per_capita_amount` is `decimal(18,12)`: only 6 digits before the decimal point (max 999,999.999999). NCR is already at 631,279 in 2025. | Use a wider decimal (for example `decimal(18,4)`) or DOUBLE in Silver, so later years do not overflow. |
| grdp, grdp_per_capita | Bronze has 2 extra columns that are not in the Excel file: `_source_file` (string) and `_ingested_at` (timestamp) | Keep as load metadata. `Exact Duplicate Rows` includes them, so a row loaded twice at different times would not show there. `Composite Key Duplicates` would catch it (0). |
| grdp, grdp_per_capita | `source_name`, `source_file` and `_source_file` all hold a file name (`source_name` = `source_file` in the Excel file) | Keep one. Confirm `_source_file` has the same value. |
| grdp vs grdp_per_capita | Column order differs: `price_basis` and `year` are in different positions | Use the same column order in both Silver tables. |

## Assumptions to Verify
| Interpretation / Question | Evidence | Status |
|---|---|---|
| Bronze has the same rows and values as the Excel files in the Volume | 114 rows on both sides, 0 rows on only one side, 0 differences in `region_id`, `geographic_level`, `unit` and the amount | Confirmed |
| `IV-B` is the right code for MIMAROPA and `PH` for the national row | Not in the data. `IV-B` is the usual PSA code. | Pending |
| `GRDP_amount` is in thousand pesos, `GRDP_per_capita_amount` in pesos per person | One `unit` value per table. Population implied by the two tables is realistic (Philippines 111.9M in 2023). | Supported |
| Population and regional spending tables use the same 18 regions | Not checked yet. | Pending |

## Semantic Notes
- GRDP measures economic output; it is not poverty incidence.
- GRDP per capita measures output per person; it is not personal income.
- Units: `GRDP_amount` = thousand Philippine pesos. `GRDP_per_capita_amount` = Philippine pesos per person.
- Price basis: `At Current Prices` and `At Constant 2018 Prices` are kept separate. Choose one per analysis.
- Do not add national totals to regional records.
- Do not sum or simply average regional GRDP per capita.

## Conclusion
- Proposed Silver rules:
  1. Fill `region_id` for MIMAROPA Region and for Philippines, fill or null.
  2. Cast `source_row` and `year` to INT. Widen the `GRDP_per_capita_amount` decimal (or use DOUBLE).
  3. Put ingestion timestamps for silver and ideally, bronze.
  
- Ready for Q3? Yes but needs to be reviewed and rechecked according to findings.
- Open questions:
  1. Do we fill in the empty `region_id` codes for MIMAROPA Region and the national row?
  2. Do the population and spending tables use the same region list?
